# Homework 9 Starter Code



In [1]:

solver = 'appsi_highs'

import pyomo.environ as pyo
SOLVER = pyo.SolverFactory(solver)

assert SOLVER.available(), f"Solver {solver} is not available."

## Q1: Best-subset regression

We implemented:


*   Function `generate_sparse_regression_data` generates sparse synthetic linear-regression data with a known true support.
*   Function `fit_ridge_cv` fits a baseline ridge regression with cross-validated lambda.
*   Function `solve_best_subset_reg` implements the best-subset regression given training data (assuming a fixed k)

Your tasks:

*   Complete the function `select_k_via_cv_MILP` to implement the Pyomo MILP model for cross validating `k` for the best-subset absolute-error regression.
*   Run `main` to see whether the optimal regression model with the cross-validated `k` contains the true support and compare the generalization error of best-subset regression against ridge regression.





In [2]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error

# =========================
# Data generator
# =========================
def generate_sparse_regression_data(
    n_train=300,
    n_test=200,
    d=20,
    support=(0, 1, 4, 8),
    beta_values=(3.0, -2.5, 1.8, -1.2),
    noise_std=1.0,
    rho=0.35,
    seed=164,
):
    """
    Generate synthetic sparse linear regression data:
        y = X w_true + b_true + epsilon

    Features are Gaussian with AR(1)-style correlation:
        Cov(X_j, X_k) = rho^{|j-k|}

    Returns
    -------
    X_train, y_train, X_test, y_test, w_true, b_true, true_support
    """
    rng = np.random.default_rng(seed)

    if len(support) != len(beta_values):
        raise ValueError("support and beta_values must have the same length")
    if d <= max(support):
        raise ValueError("d must exceed the largest support index")
    if not (0 <= rho < 1):
        raise ValueError("rho must be in [0, 1)")

    true_support = tuple(sorted(support))
    w_true = np.zeros(d)
    for j, val in zip(true_support, beta_values):
        w_true[j] = val
    b_true = 0.75

    idx = np.arange(d)
    Sigma = rho ** np.abs(np.subtract.outer(idx, idx))
    L = np.linalg.cholesky(Sigma + 1e-10 * np.eye(d))

    def sample_X(n):
        Z = rng.standard_normal((n, d))
        return Z @ L.T

    X_train = sample_X(n_train)
    X_test = sample_X(n_test)

    eps_train = noise_std * rng.standard_normal(n_train)
    eps_test = noise_std * rng.standard_normal(n_test)

    y_train = X_train @ w_true + b_true + eps_train
    y_test = X_test @ w_true + b_true + eps_test

    return X_train, y_train, X_test, y_test, w_true, b_true, true_support

In [3]:
# =========================
# Ridge baseline
# =========================
def fit_ridge_cv(X_train, y_train, X_test, y_test, alphas=None, n_splits=5, seed=164):
    """
    Baseline ridge regression with K-fold CV over alpha.
    """
    if alphas is None:
        alphas = np.logspace(-3, 2, 20)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    cv_mae = []

    for alpha in alphas:
        fold_mae = []
        for tr_idx, val_idx in kf.split(X_train):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            model = Ridge(alpha=alpha, fit_intercept=True)
            model.fit(X_tr, y_tr)
            pred_val = model.predict(X_val)
            fold_mae.append(mean_absolute_error(y_val, pred_val))

        cv_mae.append(np.mean(fold_mae))

    best_alpha = float(alphas[int(np.argmin(cv_mae))])
    best_model = Ridge(alpha=best_alpha, fit_intercept=True)
    best_model.fit(X_train, y_train)

    pred_train = best_model.predict(X_train)
    pred_test = best_model.predict(X_test)

    results = {
        "best_alpha": best_alpha,
        "cv_mae": float(np.min(cv_mae)),
        "train_mae": float(mean_absolute_error(y_train, pred_train)),
        "test_mae": float(mean_absolute_error(y_test, pred_test)),
        "train_mse": float(mean_squared_error(y_train, pred_train)),
        "test_mse": float(mean_squared_error(y_test, pred_test)),
        "coef": best_model.coef_.copy(),
        "intercept": float(best_model.intercept_),
        "model": best_model,
    }
    return results

In [4]:
# =========================
# Best subset solver (for fixed k)
# =========================
def solve_best_subset_reg(X, y, k, M=10.0, solver_name="gurobi", tee=False):
    """
    Solve:
        min_{w,b} sum_i | y_i - w^T x_i - b |
        s.t. ||w||_0 <= k

    MILP reformulation:
        residual r_i = y_i - sum_j x_ij w_j - b
        t_i >= r_i, t_i >= -r_i
        -M z_j <= w_j <= M z_j
        sum_j z_j <= k, z_j binary
    """
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    n, d = X.shape
    model = pyo.ConcreteModel()

    model.I = pyo.RangeSet(0, n - 1)
    model.J = pyo.RangeSet(0, d - 1)

    model.w = pyo.Var(model.J, domain=pyo.Reals)
    model.b = pyo.Var(domain=pyo.Reals)
    model.z = pyo.Var(model.J, domain=pyo.Binary)
    model.r = pyo.Var(model.I, domain=pyo.Reals)
    model.t = pyo.Var(model.I, domain=pyo.NonNegativeReals)

    def residual_rule(m, i):
        return m.r[i] == y[i] - sum(X[i, j] * m.w[j] for j in m.J) - m.b

    model.residual = pyo.Constraint(model.I, rule=residual_rule)
    model.abs_pos = pyo.Constraint(model.I, rule=lambda m, i: m.t[i] >= m.r[i])
    model.abs_neg = pyo.Constraint(model.I, rule=lambda m, i: m.t[i] >= -m.r[i])

    model.link_lb = pyo.Constraint(model.J, rule=lambda m, j: m.w[j] >= -M * m.z[j])
    model.link_ub = pyo.Constraint(model.J, rule=lambda m, j: m.w[j] <= M * m.z[j])

    model.cardinality = pyo.Constraint(expr=sum(model.z[j] for j in model.J) <= int(k))

    model.obj = pyo.Objective(expr=sum(model.t[i] for i in model.I), sense=pyo.minimize)

    results = SOLVER.solve(model, tee=tee)

    term_cond = str(results.solver.termination_condition).lower()
    if "optimal" not in term_cond and "feasible" not in term_cond:
        raise RuntimeError(f"Solver did not return an acceptable solution: {results.solver}")

    w_hat = np.array([pyo.value(model.w[j]) for j in model.J], dtype=float)
    b_hat = float(pyo.value(model.b))
    z_hat = np.array([round(pyo.value(model.z[j])) for j in model.J], dtype=int)
    obj = float(pyo.value(model.obj))
    selected_support = tuple(np.where(z_hat > 0.5)[0].tolist())

    return {
        "w": w_hat,
        "b": b_hat,
        "z": z_hat,
        "selected_support": selected_support,
        "objective": obj,
    }

In [17]:
# =========================
# TODO: YOUR TASK
# =========================
def select_k_via_cv_MILP(X, y, k_grid, n_splits=5, M=10.0):
    """
    Build and solve the cross-validation MILP for each fixed candidate k in k_grid,
    then return the k with the smallest average validation loss.

    YOUR TASK: complete the Pyomo model inside the loop over k.


    Parameters
    ----------
    X : ndarray of shape (n, d)
    y : ndarray of shape (n,)
    k_grid : iterable of candidate subset sizes
    n_splits : int
    M : float
        Big-M bound for coefficient linking constraints.

    Returns
    -------
    best_k : int
        The selected subset size.
    """

    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)

    n, d = X.shape
    k_list = list(k_grid)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=164)
    splits = list(kf.split(X))

    # Build index sets for training and validation observations by fold
    train_triplets = []
    val_triplets = []
    train_lookup = {}
    val_lookup = {}

    for l_idx, (tr_idx, val_idx) in enumerate(splits):
        train_lookup[l_idx] = list(tr_idx)
        val_lookup[l_idx] = list(val_idx)

        for i in tr_idx:
            train_triplets.append((l_idx, int(i)))
        for i in val_idx:
            val_triplets.append((l_idx, int(i)))

    best_k = None
    best_obj = np.inf

    for k in k_list:
        m = pyo.ConcreteModel()

        # Sets
        m.L = pyo.RangeSet(0, n_splits - 1)      # folds
        m.J = pyo.RangeSet(0, d - 1)             # features
        m.TRAIN = pyo.Set(initialize=train_triplets, dimen=2)
        m.VAL = pyo.Set(initialize=val_triplets, dimen=2)

        # -------------------------
        # Decision variables
        # -------------------------
        # Example definitions:
        #
        # One regression model per fold l:
        #   w[l,j] : coefficient for feature j in fold l
        #   b[l]   : intercept for fold l
        #   z[l,j] : whether feature j is selected in fold l
        #
        # Residual / absolute-value variables:
        #   r[l,i] : training residual for sample i in fold l
        #   t[l,i] : absolute value of training residual
        #   s[l,i] : absolute value of validation residual
        
        m.w = pyo.Var(m.L, m.J, domain=pyo.Reals)
        m.b = pyo.Var(m.L, domain=pyo.Reals)
        m.z = pyo.Var(m.L, m.J, domain=pyo.Binary)
        
        m.r = pyo.Var(m.TRAIN, domain=pyo.Reals)
        m.t = pyo.Var(m.TRAIN, domain=pyo.NonNegativeReals)
        m.s = pyo.Var(m.VAL, domain=pyo.NonNegativeReals)
        

        # -------------------------
        # Constraints
        # -------------------------

        def residuals(m, l, i):
            return m.r[l, i] == y[i] - sum(X[i, j] * m.w[l, j] for j in m.J) - m.b[l]
        
        m.residuals = pyo.Constraint(m.TRAIN, rule=residuals)
        
        
        def abs_pos_rule(m, l, i):
            return m.t[l, i] >= m.r[l, i]
        m.abs_pos = pyo.Constraint(m.TRAIN, rule=abs_pos_rule)
        
        def abs_neg_rule(m, l, i):
            return m.t[l, i] >= -m.r[l, i]
        
        m.abs_neg = pyo.Constraint(m.TRAIN, rule=abs_neg_rule)
        
        def lb_rule(m, l, j):
            return -m.w[l, j] <= M * m.z[l, j]
        m.lb = pyo.Constraint(m.L, m.J, rule=lb_rule)
        
        def ub_rule(m, l, j):
            return m.w[l, j] <= M * m.z[l, j]
        m.ub = pyo.Constraint(m.L, m.J, rule=ub_rule)
        
        
        def sparsity_rule(m, l):
            return sum(m.z[l, j] for j in m.J) <= k 
        m.sparsity = pyo.Constraint(m.L, rule=sparsity_rule)

        
        def val_residual_pos_rule(m, l, i):
            return m.s[l, i] >= y[i] - sum(X[i, j] * m.w[l, j] for j in m.J) - m.b[l]
        m.val_resuidual_pos = pyo.Constraint(m.VAL, rule=val_residual_pos_rule)
        
        
        def val_residual_neg_rule(m, l, i):
            return m.s[l, i] >= - (y[i] - sum(X[i, j] * m.w[l, j] for j in m.J) - m.b[l])
        m.val_residual_neg = pyo.Constraint(m.VAL, rule=val_residual_neg_rule)
        # -------------------------
        # Objective
        # -------------------------
        # Minimize average validation absolute error across folds:
        #
        #   (1 / n_splits) * sum_l (1 / |I_val^l|) * sum_{i in I_val^l} s[l,i]
        #
        # TODO: add the objective
        
        def objective_function(m):
            return (1/n_splits) * sum((1/len(val_lookup[l])) * sum(m.s[l, i] for i in val_lookup[l]) for l in m.L)
        m.obj = pyo.Objective(rule=objective_function, sense=pyo.minimize)

        result = SOLVER.solve(m)

        obj_val = pyo.value(m.obj)
        print(f"k = {k} -- CV error: {obj_val}")
        if obj_val < best_obj - 1e-1:
            best_obj = obj_val
            best_k = k
            
        print(f"best k so far: {best_k}")

    return best_k

In [18]:
# =========================
# Helper utilities
# =========================
def predict_linear(X, w, b):
    return X @ np.asarray(w) + float(b)


def support_from_vector(w, tol=1e-6):
    return tuple(np.where(np.abs(np.asarray(w)) > tol)[0].tolist())


def contains_true_support(selected_support, true_support):
    return set(true_support).issubset(set(selected_support))


def choose_big_m_from_data(X, y, min_M=8.0):
    x_scale = max(np.mean(np.std(X, axis=0)), 1e-6)
    y_scale = max(np.std(y), 1e-6)
    M = 2.0 * y_scale / x_scale
    return float(max(min_M, M))


# =========================
# Main
# =========================
if __name__ == "__main__":
    # Data settings
    n_train = 200
    n_test = 400
    d = 10
    true_support = (0, 1, 4, 8)
    beta_values = (4.5, -3.5, 3.0, -2.5)
    noise_std = 1.
    rho = 0.25
    seed = 164

    # Generate data
    X_train, y_train, X_test, y_test, w_true, b_true, true_support = generate_sparse_regression_data(
        n_train=n_train,
        n_test=n_test,
        d=d,
        support=true_support,
        beta_values=beta_values,
        noise_std=noise_std,
        rho=rho,
        seed=seed,
    )

    # Big-M and candidate k values
    M = choose_big_m_from_data(X_train, y_train)
    k_grid = list(range(0, 9))

    print("=== Synthetic sparse regression experiment ===")
    print(f"n_train = {n_train}, n_test = {n_test}, d = {d}")
    print(f"True support: {true_support}")
    print(f"True coefficients: {w_true}")
    print(f"True intercept: {b_true:.3f}")
    print(f"Recommended big-M: {M:.2f}")
    print(f"Candidate k values: {k_grid}")

    # Ridge baseline
    ridge_results = fit_ridge_cv(X_train, y_train, X_test, y_test)
    ridge_support = support_from_vector(ridge_results["coef"], tol=1e-4)

    print("\n=== Ridge baseline ===")
    print(f"Best alpha: {ridge_results['best_alpha']:.4f}")
    print(f"CV MAE:     {ridge_results['cv_mae']:.4f}")
    print(f"Train MAE:  {ridge_results['train_mae']:.4f}")
    print(f"Test MAE:   {ridge_results['test_mae']:.4f}")
    print(f"Train MSE:  {ridge_results['train_mse']:.4f}")
    print(f"Test MSE:   {ridge_results['test_mse']:.4f}")
    print(f"Ridge support (tol-based): {ridge_support}")

    # Your CV MILP
    print("\n=== Selecting k by CV-MILP ===")
    best_k = select_k_via_cv_MILP(X_train, y_train, k_grid=k_grid, n_splits=5, M=M)
    print(f"Selected best_k = {best_k}")

    # Refit best-subset model on full training set using selected k
    best_subset_fit = solve_best_subset_reg(X_train, y_train, k=best_k, M=M)
    w_hat = best_subset_fit["w"]
    b_hat = best_subset_fit["b"]
    selected_support = best_subset_fit["selected_support"]

    train_pred = predict_linear(X_train, w_hat, b_hat)
    test_pred = predict_linear(X_test, w_hat, b_hat)

    train_mae = mean_absolute_error(y_train, train_pred)
    test_mae = mean_absolute_error(y_test, test_pred)
    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    print("\n=== Best-subset regression with selected k ===")
    print(f"Selected support: {selected_support}")
    print(f"Contains true support? {contains_true_support(selected_support, true_support)}")
    print(f"Selected k equals true support size? {best_k == len(true_support)}")
    print(f"Train MAE: {train_mae:.4f}")
    print(f"Test MAE:  {test_mae:.4f}")
    print(f"Train MSE: {train_mse:.4f}")
    print(f"Test MSE:  {test_mse:.4f}")

    print("\n=== Comparison against ridge ===")
    print(f"Best-subset better than ridge on test MAE? {test_mae < ridge_results['test_mae']}")
    print(f"Best-subset better than ridge on test MSE? {test_mse < ridge_results['test_mse']}")

=== Synthetic sparse regression experiment ===
n_train = 200, n_test = 400, d = 10
True support: (0, 1, 4, 8)
True coefficients: [ 4.5 -3.5  0.   0.   3.   0.   0.   0.  -2.5  0. ]
True intercept: 0.750
Recommended big-M: 13.36
Candidate k values: [0, 1, 2, 3, 4, 5, 6, 7, 8]

=== Ridge baseline ===
Best alpha: 0.7848
CV MAE:     0.8667
Train MAE:  0.7927
Test MAE:   0.8172
Train MSE:  0.9883
Test MSE:   1.0259
Ridge support (tol-based): (0, 1, 2, 3, 4, 5, 6, 7, 8, 9)

=== Selecting k by CV-MILP ===
k = 0 -- CV error: 5.060349886498095
best k so far: 0
k = 1 -- CV error: 3.742933279466987
best k so far: 1
k = 2 -- CV error: 2.898949242685033
best k so far: 2
k = 3 -- CV error: 1.8125203493705098
best k so far: 3
k = 4 -- CV error: 0.7099657161825506
best k so far: 4
k = 5 -- CV error: 0.6655181834301537
best k so far: 4
k = 6 -- CV error: 0.6482196710908569
best k so far: 4
k = 7 -- CV error: 0.6290896529087897
best k so far: 4
k = 8 -- CV error: 0.6153774425855625
best k so far: 4
Sele

## Q2: Scheduling under random no-shows and service time

In [1]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo


def sample_discrete_service_and_noshow(
    num_scenarios,
    slow_times,
    fast_times,
    p_no_show,
    p_slow,
    p_fast,
    seed=123,
):
    """
    Outcomes:
      0 = no-show
      1 = slow
      2 = fast

    Returns
    -------
    service : ndarray, shape (S, n)
        Realized service times before assistant reduction.
    show : ndarray, shape (S, n)
        0/1 indicator of whether patient shows.
    outcome : ndarray, shape (S, n)
        Encoded outcome in {0,1,2}.
    """
    rng = np.random.default_rng(seed)

    slow_times = np.asarray(slow_times, dtype=float)
    fast_times = np.asarray(fast_times, dtype=float)
    p_no_show = np.asarray(p_no_show, dtype=float)
    p_slow = np.asarray(p_slow, dtype=float)
    p_fast = np.asarray(p_fast, dtype=float)

    n = len(slow_times)
    outcome = np.zeros((num_scenarios, n), dtype=int)

    for i in range(n):
        probs = np.array([p_no_show[i], p_slow[i], p_fast[i]], dtype=float)
        probs = probs / probs.sum()
        outcome[:, i] = rng.choice([0, 1, 2], size=num_scenarios, p=probs)

    show = (outcome != 0).astype(int)
    service = np.where(outcome == 1, slow_times, 0.0)
    service = np.where(outcome == 2, fast_times, service)

    return service, show, outcome


def build_history_groups(outcome_matrix):
    """
    Group scenarios by identical realized history.

    For stage i, the history is outcome[:, :i-1], since decision y_i
    can depend only on outcomes up to patient i-1.

    Returns
    -------
    history_groups : dict
        history_groups[i][history_tuple] = list of scenario indices
        for i = 2,...,n
    """
    S, n = outcome_matrix.shape
    history_groups = {}

    for i in range(2, n + 1):
        groups = {}
        for s in range(S):
            hist = tuple(outcome_matrix[s, : i - 1].tolist())
            groups.setdefault(hist, []).append(s)
        history_groups[i] = groups

    return history_groups


def solve_schedule_with_no_shows(
    service_matrix,
    show_matrix,
    outcome_matrix,
    T,
    cw,
    cu,
    co,
    ca,
    gamma,
    enforce_nonanticipativity=True,
    solver_name="appsi_highs",
    tee=False,
):
    """
    Solve the adaptive scheduling model with random no-shows and discrete service modes.

    Key modeling choices:
      - realized base service = q_i * s_i
      - effective service     = q_i*s_i - gamma_i * y_i
      - assistant only if show: y_i <= q_i
      - waiting penalty counts only for shown patients

    Returns
    -------
    result : dict
        Includes x, y, objective, and a full report of metrics.
    """
    service_matrix = np.asarray(service_matrix, dtype=float)
    show_matrix = np.asarray(show_matrix, dtype=int)
    outcome_matrix = np.asarray(outcome_matrix, dtype=int)

    S, n = service_matrix.shape
    cw = np.asarray(cw, dtype=float)
    cu = np.asarray(cu, dtype=float)
    ca = np.asarray(ca, dtype=float)
    gamma = np.asarray(gamma, dtype=float)

    if len(cw) != n or len(cu) != n or len(ca) != n or len(gamma) != n:
        raise ValueError("cw, cu, ca, gamma must all have length n")

    # realized base service = q_i * s_i
    base_service = show_matrix * service_matrix

    history_groups = build_history_groups(outcome_matrix)

    m = pyo.ConcreteModel()
    m.O = pyo.RangeSet(0, S - 1)      # scenarios
    m.I = pyo.RangeSet(1, n)          # patients
    m.J = pyo.RangeSet(1, n - 1)      # schedule gaps
    m.K = pyo.RangeSet(2, n)          # adaptive stages

    # First-stage schedule
    m.x = pyo.Var(m.J, domain=pyo.NonNegativeReals)

    # Adaptive decisions / state variables
    m.y = pyo.Var(m.O, m.K, domain=pyo.Binary)
    m.sbar = pyo.Var(m.O, m.I, domain=pyo.NonNegativeReals)
    m.w = pyo.Var(m.O, m.I, domain=pyo.NonNegativeReals)
    m.u = pyo.Var(m.O, m.I, domain=pyo.NonNegativeReals)
    m.W = pyo.Var(m.O, domain=pyo.NonNegativeReals)

    # Feasible schedule length
    m.total_length = pyo.Constraint(expr=sum(m.x[j] for j in m.J) <= T)

    # Initial condition
    m.first_wait = pyo.Constraint(m.O, rule=lambda m, o: m.w[o, 1] == 0.0)

    # Effective service for patient 1 is fixed
    m.first_service = pyo.Constraint(
        m.O, rule=lambda m, o: m.sbar[o, 1] == base_service[o, 0]
    )

    # Effective service for patients 2,...,n
    def sbar_rule(m, o, i):
        return m.sbar[o, i] == int(show_matrix[o, i - 1]) * base_service[o, i - 1] - gamma[i - 1] * m.y[o, i]
    m.sbar_def = pyo.Constraint(m.O, m.K, rule=sbar_rule)

    # # If patient i does not show, cannot use assistant
    # def show_logic_rule(m, o, i):
    #     return m.y[o, i] <= int(show_matrix[o, i - 1])
    # m.show_logic = pyo.Constraint(m.O, m.K, rule=show_logic_rule)

    # Waiting / idle recursion
    def rec_rule(m, o, i):
        if i == 1:
            return pyo.Constraint.Skip
        return m.w[o, i] - m.u[o, i - 1] == m.sbar[o, i - 1] + m.w[o, i - 1] - m.x[i - 1]
    m.rec = pyo.Constraint(m.O, m.I, rule=rec_rule)

    # Overtime equation
    def overtime_rule(m, o):
        return m.W[o] - m.u[o, n] == m.sbar[o, n] + m.w[o, n] + sum(m.x[j] for j in m.J) - T
    m.overtime = pyo.Constraint(m.O, rule=overtime_rule)

    # Non-anticipativity by history groups
    if enforce_nonanticipativity:
        m.nonant = pyo.ConstraintList()
        for i in range(2, n + 1):
            for _, scen_list in history_groups[i].items():
                root = scen_list[0]
                for o in scen_list[1:]:
                    m.nonant.add(m.y[root, i] == m.y[o, i])

    # Uniform SAA weights
    prob = 1.0 / S

    # Objective:
    # waiting penalty should only apply if patient shows
    m.obj = pyo.Objective(
        expr=sum(
            prob * (
                sum(cw[i - 1] * show_matrix[o, i - 1] * m.w[o, i] for i in m.I)
                + sum(cu[i - 1] * m.u[o, i] for i in m.I)
                + sum(ca[i - 1] * m.y[o, i] for i in m.K)
                + co * m.W[o]
            )
            for o in m.O
        ),
        sense=pyo.minimize,
    )

    solver = pyo.SolverFactory(solver_name)
    if solver is None or not solver.available(False):
        raise RuntimeError(f"Solver '{solver_name}' is not available.")

    res = solver.solve(m, tee=tee)

    # ---------- Extract solution ----------
    x_sol = np.array([pyo.value(m.x[j]) for j in m.J], dtype=float)

    y_sol = np.zeros((S, n), dtype=int)
    for o in range(S):
        for i in range(1, n + 1):
            if i == 1:
                y_sol[o, i - 1] = 0
            else:
                y_sol[o, i - 1] = int(round(pyo.value(m.y[o, i])))

    w_sol = np.array([[pyo.value(m.w[o, i]) for i in m.I] for o in m.O], dtype=float)
    u_sol = np.array([[pyo.value(m.u[o, i]) for i in m.I] for o in m.O], dtype=float)
    W_sol = np.array([pyo.value(m.W[o]) for o in m.O], dtype=float)
    sbar_sol = np.array([[pyo.value(m.sbar[o, i]) for i in m.I] for o in m.O], dtype=float)

    # ---------- Report metrics ----------
    wait_cost_by_s = np.sum(cw * show_matrix * w_sol, axis=1)
    idle_cost_by_s = np.sum(cu * u_sol, axis=1)
    assist_cost_by_s = np.sum(ca * y_sol, axis=1)
    overtime_cost_by_s = co * W_sol
    total_cost_by_s = wait_cost_by_s + idle_cost_by_s + assist_cost_by_s + overtime_cost_by_s

    shown_count_by_s = np.sum(show_matrix, axis=1)
    wait_total_by_s = np.sum(show_matrix * w_sol, axis=1)
    idle_total_by_s = np.sum(u_sol, axis=1)
    assist_total_by_s = np.sum(y_sol, axis=1)

    expected_total_wait = float(np.mean(wait_total_by_s))
    expected_total_idle = float(np.mean(idle_total_by_s))
    expected_overtime = float(np.mean(W_sol))
    expected_assist_usage = float(np.mean(assist_total_by_s))
    expected_shown_patients = float(np.mean(shown_count_by_s))

    avg_wait_per_patient = expected_total_wait / n
    avg_idle_per_patient = expected_total_idle / n
    avg_wait_per_shown_patient = (
        expected_total_wait / expected_shown_patients if expected_shown_patients > 1e-8 else 0.0
    )

    stage_assist_rate = {
        i: float(np.mean(y_sol[:, i - 1])) for i in range(2, n + 1)
    }
    stage_show_rate = {
        i: float(np.mean(show_matrix[:, i - 1])) for i in range(1, n + 1)
    }
    stage_wait = {
        i: float(np.mean(show_matrix[:, i - 1] * w_sol[:, i - 1])) for i in range(1, n + 1)
    }
    stage_idle = {
        i: float(np.mean(u_sol[:, i - 1])) for i in range(1, n + 1)
    }

    report = {
        "objective": float(pyo.value(m.obj)),
        "expected_wait_cost": float(np.mean(wait_cost_by_s)),
        "expected_idle_cost": float(np.mean(idle_cost_by_s)),
        "expected_assist_cost": float(np.mean(assist_cost_by_s)),
        "expected_overtime_cost": float(np.mean(overtime_cost_by_s)),
        "expected_total_wait": expected_total_wait,
        "expected_total_idle": expected_total_idle,
        "expected_overtime": expected_overtime,
        "expected_assist_usage": expected_assist_usage,
        "expected_shown_patients": expected_shown_patients,
        "avg_wait_per_patient": float(avg_wait_per_patient),
        "avg_idle_per_patient": float(avg_idle_per_patient),
        "avg_wait_per_shown_patient": float(avg_wait_per_shown_patient),
        "stage_assist_rate": stage_assist_rate,
        "stage_show_rate": stage_show_rate,
        "stage_expected_wait": stage_wait,
        "stage_expected_idle": stage_idle,
    }

    scenario_df = pd.DataFrame({
        "scenario": list(range(S)),
        "shown_patients": shown_count_by_s,
        "total_wait": wait_total_by_s,
        "total_idle": idle_total_by_s,
        "overtime": W_sol,
        "assist_uses": assist_total_by_s,
        "wait_cost": wait_cost_by_s,
        "idle_cost": idle_cost_by_s,
        "assist_cost": assist_cost_by_s,
        "overtime_cost": overtime_cost_by_s,
        "total_cost": total_cost_by_s,
    })

    return {
        "x": x_sol,
        "y": y_sol,
        "w": w_sol,
        "u": u_sol,
        "W": W_sol,
        "sbar": sbar_sol,
        "objective": float(pyo.value(m.obj)),
        "report": report,
        "scenario_metrics": scenario_df,
        "model": m,
        "solve_result": res,
    }


def compare_nonanticipativity(
    num_scenarios=250,
    n=10,
    T=8.0,
    seed=123,
    solver_name="appsi_highs",
):
    """
    Small driver to compare enforce_nonanticipativity=True vs False.
    """
    # Example synthetic instance
    slow_times = np.linspace(1.3, 2.2, n)
    fast_times = np.linspace(0.7, 1.2, n)

    p_no_show = np.full(n, 0.10)
    p_slow = np.full(n, 0.55)
    p_fast = np.full(n, 0.35)

    cw = np.full(n, 2.0)
    cu = np.full(n, 0.5)
    ca = np.array([0.0] + [0.8] * (n - 1))   # ca[0] unused
    gamma = np.array([0.0] + [0.25] * (n - 1))  # gamma[0] unused
    co = 6.0

    service, show, outcome = sample_discrete_service_and_noshow(
        num_scenarios=num_scenarios,
        slow_times=slow_times,
        fast_times=fast_times,
        p_no_show=p_no_show,
        p_slow=p_slow,
        p_fast=p_fast,
        seed=seed,
    )

    sol_na = solve_schedule_with_no_shows(
        service, show, outcome,
        T=T, cw=cw, cu=cu, co=co, ca=ca, gamma=gamma,
        enforce_nonanticipativity=True,
        solver_name=solver_name,
    )

    sol_relaxed = solve_schedule_with_no_shows(
        service, show, outcome,
        T=T, cw=cw, cu=cu, co=co, ca=ca, gamma=gamma,
        enforce_nonanticipativity=False,
        solver_name=solver_name,
    )

    comparison = pd.DataFrame({
        "metric": [
            "objective",
            # "expected_wait_cost",
            # "expected_idle_cost",
            # "expected_assist_cost",
            # "expected_overtime_cost",
            "expected_total_wait",
            "expected_total_idle",
            "expected_overtime",
            "expected_assist_usage",
            "avg_wait_per_shown_patient",
        ],
        "NA_enforced": [
            sol_na["report"]["objective"],
            # sol_na["report"]["expected_wait_cost"],
            # sol_na["report"]["expected_idle_cost"],
            # sol_na["report"]["expected_assist_cost"],
            # sol_na["report"]["expected_overtime_cost"],
            sol_na["report"]["expected_total_wait"],
            sol_na["report"]["expected_total_idle"],
            sol_na["report"]["expected_overtime"],
            sol_na["report"]["expected_assist_usage"],
            sol_na["report"]["avg_wait_per_shown_patient"],
        ],
        "NA_relaxed": [
            sol_relaxed["report"]["objective"],
            # sol_relaxed["report"]["expected_wait_cost"],
            # sol_relaxed["report"]["expected_idle_cost"],
            # sol_relaxed["report"]["expected_assist_cost"],
            # sol_relaxed["report"]["expected_overtime_cost"],
            sol_relaxed["report"]["expected_total_wait"],
            sol_relaxed["report"]["expected_total_idle"],
            sol_relaxed["report"]["expected_overtime"],
            sol_relaxed["report"]["expected_assist_usage"],
            sol_relaxed["report"]["avg_wait_per_shown_patient"],
        ],
    })
    comparison["gap_NA_minus_relaxed"] = comparison["NA_enforced"] - comparison["NA_relaxed"]

    return {
        "NA_enforced": sol_na,
        "NA_relaxed": sol_relaxed,
        "comparison": comparison,
    }


if __name__ == "__main__":
    out = compare_nonanticipativity(
        num_scenarios=200,
        n=6,
        T=8.0,
        seed=164,
        solver_name="appsi_highs",
    )

    print("=== Comparison of objective and metrics ===")
    print(out["comparison"].round(4).to_string(index=False))

    print("\n=== Optimal first-stage schedule with NA enforced ===")
    print(np.round(out["NA_enforced"]["x"], 3))

    print("\n=== Optimal first-stage schedule with NA relaxed ===")
    print(np.round(out["NA_relaxed"]["x"], 3))

    print("\n=== Stagewise assistant usage rate (NA enforced) ===")
    print(out["NA_enforced"]["report"]["stage_assist_rate"])

    print("\n=== Stagewise assistant usage rate (NA relaxed) ===")
    print(out["NA_relaxed"]["report"]["stage_assist_rate"])

=== Comparison of objective and metrics ===
                    metric  NA_enforced  NA_relaxed  gap_NA_minus_relaxed
                 objective       7.6437      5.6049                2.0388
       expected_total_wait       1.5155      1.1648                0.3507
       expected_total_idle       1.0380      0.9830                0.0550
         expected_overtime       0.5743      0.1693                0.4050
     expected_assist_usage       0.8100      2.2100               -1.4000
avg_wait_per_shown_patient       0.2806      0.2157                0.0649

=== Optimal first-stage schedule with NA enforced ===
[0.7  1.42 1.41 1.25 1.69]

=== Optimal first-stage schedule with NA relaxed ===
[0.7  1.23 1.41 1.25 1.46]

=== Stagewise assistant usage rate (NA enforced) ===
{2: 0.0, 3: 0.005, 4: 0.12, 5: 0.21, 6: 0.475}

=== Stagewise assistant usage rate (NA relaxed) ===
{2: 0.485, 3: 0.305, 4: 0.425, 5: 0.415, 6: 0.58}


### Definition of Reported Metrics

Let expectations be taken over the sampled scenarios.

- **objective**: Total expected cost.

- **expected_total_wait**: Expected total waiting time across all patients,  

- **expected_total_idle**: Expected total idle time of the doctor,  

- **expected_overtime**: Expected overtime duration,  

- **avg_wait_per_shown_patient**: Average waiting time among patients who show up,

- **expected_assist_usage**: Expected total number of times the assistant is used,  $
  \mathbb{E}\left[\sum_{i=2}^n y_i\right].
  $

- **stage_assist_rate[i]**: Probability that the assistant is used at stage \(i\),  $
  \mathbb{E}[y_i].
  $